# Query Iceberg Orders Table via DuckDB

Reads the `demo.orders` Iceberg table written by the Flink job from MinIO.

**Prerequisites:** the Docker Compose stack must be running (`docker compose up -d`).

In [37]:
import duckdb

con = duckdb.connect()

# Install extensions once, then load them
con.execute("INSTALL httpfs")
con.execute("INSTALL iceberg")
con.execute("LOAD httpfs")
con.execute("LOAD iceberg")

In [38]:
# Point DuckDB at the local MinIO instance
# MinIO is exposed on localhost:9000 from the host machine
con.execute("""
CREATE OR REPLACE SECRET minio (
    TYPE      s3,
    KEY_ID    'minioadmin',
    SECRET    'minioadmin',
    ENDPOINT  'localhost:9000',
    URL_STYLE 'path',
    USE_SSL   false,
    REGION    'us-east-1'
)
""")

In [42]:
# Find the latest metadata file — handles whatever path Nessie/Iceberg chose
files = con.execute("""
    SELECT file
    FROM glob('s3://iceberg-warehouse/**/metadata/*.metadata.json')
    ORDER BY file DESC
    LIMIT 10
""").df()

METADATA_FILE = files["file"].iloc[0]
print(f"Latest metadata: {METADATA_FILE}")
files

Latest metadata: s3://iceberg-warehouse/demo/orders_cbee4291-c144-4afe-bd93-f8222d9b4b07/metadata/00001-2405a68f-e41d-4a48-a5e9-e01330fbd39f.metadata.json


,file
0,s3://iceberg-warehouse/demo/orders_cbee4291-c1...
1,s3://iceberg-warehouse/demo/orders_cbee4291-c1...


In [43]:
# Select all rows from the Iceberg table
df = con.execute(f"""
    SELECT *
    FROM iceberg_scan('{METADATA_FILE}')
    ORDER BY event_time DESC
""").df()
df

,order_id,customer_id,product_id,quantity,unit_price,status,event_time
0,ord-000117,cust-035,prod-001,8,105.55,CANCELLED,2026-04-23 19:54:31.964
1,ord-000116,cust-045,prod-018,7,19.02,PLACED,2026-04-23 19:54:31.461
2,ord-000115,cust-030,prod-016,3,161.60,DELIVERED,2026-04-23 19:54:30.958
3,ord-000114,cust-044,prod-007,6,331.52,SHIPPED,2026-04-23 19:54:30.455
4,ord-000113,cust-041,prod-002,7,179.86,CONFIRMED,2026-04-23 19:54:29.953
...,...,...,...,...,...,...,...
12911,ord-000005,cust-033,prod-012,6,485.52,PLACED,2026-04-22 18:52:11.215
12912,ord-000004,cust-039,prod-011,6,481.45,PLACED,2026-04-22 18:52:10.712
12913,ord-000003,cust-017,prod-002,7,27.84,CONFIRMED,2026-04-22 18:52:10.208
12914,ord-000002,cust-025,prod-014,7,256.65,PLACED,2026-04-22 18:52:09.703


In [33]:
# Row count and basic stats
con.execute(f"""
    SELECT
        count(*)                             AS total_rows,
        count(DISTINCT customer_id)          AS unique_customers,
        count(DISTINCT product_id)           AS unique_products,
        round(sum(quantity * unit_price), 2) AS total_revenue,
        min(event_time)                      AS earliest_event,
        max(event_time)                      AS latest_event
    FROM iceberg_scan('{METADATA_FILE}')
""").df()

,total_rows,unique_customers,unique_products,total_revenue,earliest_event,latest_event
0,0,0,0,NaN,NaT,NaT


In [ ]:
# Orders by status
con.execute(f"""
    SELECT
        status,
        count(*)                             AS order_count,
        round(sum(quantity * unit_price), 2) AS revenue
    FROM iceberg_scan('{METADATA_FILE}')
    GROUP BY status
    ORDER BY order_count DESC
""").df()

In [ ]:
# Snapshot history
con.execute(f"""
    SELECT *
    FROM iceberg_snapshots('{METADATA_FILE}')
""").df()